In [0]:
USE CATALOG fraud_lakehouse;

In [0]:
SELECT
  timestamp,
  origin.flow_name,
  details:flow_progress.metrics.num_output_rows AS output_rows,
  details:flow_progress.data_quality.dropped_records AS dropped,
  details:flow_progress.data_quality.expectations AS expectations
FROM event_log(TABLE(silver.transactions))
WHERE event_type = 'flow_progress'
  AND details:flow_progress.data_quality IS NOT NULL
ORDER BY timestamp DESC
LIMIT 20

In [0]:
SELECT
  timestamp,
  flow_name,
  e.name AS expectation,
  e.passed_records,
  e.failed_records
FROM (
  SELECT
    timestamp,
    origin.flow_name AS flow_name,
    explode(from_json(
      details:flow_progress.data_quality.expectations,
      'array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>'
    )) AS e
  FROM event_log(TABLE(silver.transactions))
  WHERE event_type = 'flow_progress'
    AND details:flow_progress.data_quality.expectations IS NOT NULL
)
ORDER BY timestamp DESC, flow_name, expectation

In [0]:
CREATE OR REPLACE VIEW fraud_lakehouse.gold.pipeline_quality_log AS
SELECT
  event_time,
  to_date(event_time) AS run_date,
  update_id,
  flow_name,
  e.name AS expectation,
  e.passed_records,
  e.failed_records,
  round(100 * e.failed_records / nullif(e.passed_records + e.failed_records, 0), 2) AS failed_pct
FROM (
  SELECT
    timestamp AS event_time,
    origin.update_id AS update_id,
    origin.flow_name AS flow_name,
    explode(from_json(
      details:flow_progress.data_quality.expectations,
      'array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>'
    )) AS e
  FROM event_log(TABLE(fraud_lakehouse.silver.transactions))
  WHERE event_type = 'flow_progress'
    AND details:flow_progress.data_quality.expectations IS NOT NULL
)

In [0]:
SELECT run_date, flow_name, expectation, passed_records, failed_records, failed_pct
FROM gold.pipeline_quality_log
ORDER BY run_date DESC, failed_pct DESC

In [0]:
SELECT
  flow_name,
  expectation,
  sum(passed_records) AS passed,
  sum(failed_records) AS failed
FROM gold.pipeline_quality_log
GROUP BY flow_name, expectation
ORDER BY failed DESC

In [0]:
SELECT coalesce(max(failed_pct), 0) AS worst_rule_failed_pct
FROM gold.pipeline_quality_log
WHERE run_date >= current_date() - 1

In [0]:
SELECT round(100 * avg(cast(is_flagged AS INT)), 2) AS flagged_pct
FROM ml.fraud_scores
WHERE date_key = (SELECT max(date_key) FROM ml.fraud_scores)

In [0]:
SELECT round((unix_timestamp(current_timestamp()) - unix_timestamp(max(_ingested_at))) / 3600, 1) AS hours_since_last_load
FROM bronze.transactions